In [1]:
import ollama
import weave
import langchain
import os
from ollama_func import *
os.chdir('..')
# os.chdir(os.path.dirname(os.path.abspath(__file__)))

from pdf_parsing import parse_pdf, pdf2html, scrape_html
import markdownify

In [2]:
# Define the file path and temporary directory
file_path = "data/allshoes_catalogus/pdfs/allshoes_catalogus.pdf"
temp_dir = "tmp/allshoes_catalogus/pdfs"

# Create the temporary directory if it does not exist
if not os.path.exists(temp_dir):
    os.makedirs(temp_dir)

# Check if the temporary directory is empty
if len(os.listdir(temp_dir)) == 0:
    print(f"The directory {temp_dir} is empty.")

    # Split the PDF into individual pages
    split_paths = pdf2html.split_files(file_path, temp_dir)

    # Convert the split PDF pages to HTML
    html_paths = pdf2html.pdf2html(split_paths)

    # Scrape the HTML files to extract data into a pandas dataframe
    new_df = scrape_html.scrape_html(html_paths)

    # Convert the dataframe to a txt file and save it
    new_df.to_csv(f'{temp_dir}/pdftron.txt', sep='\t', index=False)

    # Convert HTML files to Markdown and combine them
    markdown_content = ''
    for html_file in html_paths:
        with open(html_file, 'r', encoding='utf-8') as f:
            html = f.read()
        # Convert HTML to Markdown with headers and tables
        md = markdownify.markdownify(html, heading_style="ATX")
        markdown_content += md + '\n'
    docs = markdown_content

    # Save the combined markdown content to a file
    with open(f'{temp_dir}PDFtron.md', 'w', encoding='utf-8') as md_file:
        md_file.write(markdown_content)

    print('Markdown file generated from HTML files.')

with(open(f'{temp_dir}/PDFtron.md', 'r')) as f:
    docs = f.read()
    print('Already processed')

Already processed


In [3]:
with open("prompt.txt" , "r") as f:
    prompt = f.read()
prompt += """
### Context:
{context} 

### Question:
{question}

### Response:

"""

In [ ]:
from langchain_ollama.llms import OllamaLLM
from langchain import PromptTemplate

llm = OllamaLLM(model="orca-mini", temperature=0)
embed = load_embedding_model(model_path="all-MiniLM-L6-v2")

/vol/csedu-nobackup/project/lzandbergen-storage/lars-zandbergen-pdf-extraction/models/ollama_func.py:38: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  return HuggingFaceEmbeddings(
/vol/csedu-nobackup/project/lzandbergen-storage/lars-zandbergen-pdf-extraction/.linux/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
docs = load_pdf_data("data/allshoes_catalogus/pdfs/allshoes_catalogus.pdf")
documents = split_docs(docs)
vectorstore = create_embeddings(documents, embed)

retriever = vectorstore.as_retriever()

/vol/csedu-nobackup/project/lzandbergen-storage/lars-zandbergen-pdf-extraction/.linux/lib/python3.10/site-packages/langchain_community/document_loaders/parsers/pdf.py:300: UserWarning: Warning: Empty content on page 0 of document data/allshoes_catalogus/pdfs/allshoes_catalogus.pdf
  warnings.warn(
/vol/csedu-nobackup/project/lzandbergen-storage/lars-zandbergen-pdf-extraction/.linux/lib/python3.10/site-packages/langchain_community/document_loaders/parsers/pdf.py:300: UserWarning: Warning: Empty content on page 11 of document data/allshoes_catalogus/pdfs/allshoes_catalogus.pdf
  warnings.warn(


In [ ]:
generated_prompt = PromptTemplate.from_template(prompt)
chain = load_qa_chain(retriever, llm, generated_prompt)

In [ ]:
weave.init('ollama')
get_response("Give me the product information for id: 25111", chain)


ValueError: Missing some input keys: {'\n    "feature_name_1"'}